In [18]:
import json
import os

# Configuration - manually specify file paths and their corresponding result keys
file_result_mappings = [
    # Example: {"file_path": "../results/retrieval_keywords_file1.json", "result_key": "model1"},
    # Add your mappings here
    {"file_path": "../results/retrieval_keywords_ollama:phi4_mstro.json", "result_key": "ollama:phi4"}
]

dataset = "mstro"
path_queries = f"../dataset/{dataset}/queries.jsonl"
path_id2queries = f"../dataset/{dataset}/id2queries.json"

# Load existing id2queries if it exists
if os.path.exists(path_id2queries):
    with open(path_id2queries, 'r') as f:
        id2queries = json.load(f)
    print(f"Loaded existing id2queries file with {len(id2queries)} entries")
else:
    # Initialize new file and add raw queries
    id2queries = {}
    print("Creating new id2queries file")
    
    # Only add raw queries when creating a new file
    print("Adding raw queries to new id2queries...")
    with open(path_queries, 'r') as file:
        for line in file:
            query = json.loads(line)
            id = query["_id"]
            if id not in id2queries:
                id2queries[id] = {}
            id2queries[id]["raw"] = query["text"]

# Process each mapping in the configuration
for mapping in file_result_mappings:
    file_path = mapping["file_path"]
    result_key = mapping["result_key"]
    
    if not os.path.exists(file_path):
        print(f"Warning: File {file_path} does not exist. Skipping.")
        continue
        
    print(f"Processing {file_path} with result key: {result_key}")
    
    # Load retrieval keywords
    with open(file_path, "r") as f:
        result_retrieval_keywords = json.load(f)
    
    # Add or overwrite retrieval keywords in id2queries
    new_ids_count = 0
    updated_entries_count = 0
    
    for id in result_retrieval_keywords:
        if id not in id2queries:
            # Create new ID entry if it doesn't exist
            id2queries[id] = {}
            new_ids_count += 1
            
        # Always add/update the mapping for this ID
        id2queries[id][result_key] = result_retrieval_keywords[id]
        updated_entries_count += 1
    
    print(f"Added {result_key} to {updated_entries_count} entries (including {new_ids_count} new IDs)")

print(f"Final id2queries contains {len(id2queries)} entries")
json.dump(id2queries, open(path_id2queries, "w"), indent=2, ensure_ascii=False)
print(f"Saved updated id2queries file to {path_id2queries}")

Creating new id2queries file
Adding raw queries to new id2queries...
Processing ../results/retrieval_keywords_ollama:phi4_mstro.json with result key: ollama:phi4
Added ollama:phi4 to 20 entries (including 0 new IDs)
Final id2queries contains 20 entries
Saved updated id2queries file to ../dataset/mstro/id2queries.json
